In [ ]:
# In[1]:

import json
from datetime import datetime
import pytz

# Load the task.json file
task_directory = 'rca100/cases/t069'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

# Extract alert context
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')

# Convert alert window timestamps to UTC+8 timezone
timezone = pytz.timezone('Asia/Shanghai')
alert_start = datetime.fromisoformat(alert_window['start']).astimezone(timezone)
alert_end = datetime.fromisoformat(alert_window['end']).astimezone(timezone)

alert_title, alert_start, alert_end, alert_entity

```
Out[1]:
```
The alert is titled "product-catalog 流量下跌告警" (product-catalog traffic drop alert). It occurred between 2026-04-13 04:59:52 and 2026-04-13 05:05:32 (UTC+8). The entity that triggered the alert is the operation `product-catalog::oteldemo.ProductCatalogService/ListProducts` of type `apm.operation` in the `apm` domain.

The original code execution output of IPython Kernel is also provided below for reference:

('product-catalog 流量下跌告警', datetime.datetime(2026, 4, 13, 4, 59, 52, tzinfo=<DstTzInfo 'Asia/Shanghai' CST+8:00:00 STD>), datetime.datetime(2026, 4, 13, 5, 5, 32, tzinfo=<DstTzInfo 'Asia/Shanghai' CST+8:00:00 STD>), {'entity_id': '455f921f0afc16175724d9f27faf18e4', 'entity_name': 'product-catalog::oteldemo.ProductCatalogService/ListProducts', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

# Load the topology.json file
topology_file_path = f'{task_directory}/topology.json'

with open(topology_file_path, 'r') as file:
    topology_data = json.load(file)

# Extract entities and edges
entities = topology_data.get('entities', [])
edges = topology_data.get('edges', [])

# Identify relationships involving the product-catalog service
product_catalog_service = 'product-catalog'
related_edges = [edge for edge in edges if product_catalog_service in edge['src'] or product_catalog_service in edge['dst']]

# Display the related edges
related_edges

```
Out[2]:
```
The service dependency graph does not show any direct relationships between the `product-catalog` service and other services.

The original code execution output of IPython Kernel is also provided below for reference:

[]```
```

In [ ]:
# In[3]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# Filter metrics for the product-catalog service and its operations
product_catalog_metrics = metrics_data[
    (metrics_data['entity_name'].str.contains('product-catalog', na=False)) &
    (metrics_data['domain'] == 'apm')
]

# List all available KPIs (unique metric names) for the product-catalog service and its operations
product_catalog_kpis = product_catalog_metrics['metric'].unique()

product_catalog_kpis

```
Out[3]:
```
The available KPIs for the `product-catalog` service and its operations are: `request_count`, `avg_request_latency_seconds`, `cpu`, `latency`, `mem`, and `workload`.

The original code execution output of IPython Kernel is also provided below for reference:

['request_count' 'avg_request_latency_seconds' 'cpu' 'latency' 'mem'
 'workload']```
```

In [ ]:
# In[4]:

# Ensure 'time' and 'value' columns are numeric for calculations
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Filter metrics for the product-catalog service
product_catalog_service_metrics = metrics_data[
    (metrics_data['entity_name'].str.contains('product-catalog', na=False)) &
    (metrics_data['entity_set'] == 'apm.service.legacy')
]

# Calculate global P95 thresholds for each KPI
kpi_p95_thresholds = product_catalog_service_metrics.groupby('metric')['value'].quantile(0.95).reset_index()
kpi_p95_thresholds.columns = ['metric', 'p95_threshold']

kpi_p95_thresholds

```
Out[4]:
```
The global P95 thresholds for the `product-catalog` service KPIs are as follows:
- `cpu`: 0.02295
- `latency`: 0.0000907 seconds
- `mem`: 41,542,310 bytes
- `workload`: 17,871 requests

The original code execution output of IPython Kernel is also provided below for reference:

metric  p95_threshold
0       cpu   2.295394e-02
1   latency   9.071836e-05
2       mem   4.154231e+07
3  workload   1.787100e+04```
```

In [ ]:
# In[5]:

# Convert alert time window to microseconds for filtering
alert_start_us = int(alert_start.timestamp() * 1_000_000)
alert_end_us = int(alert_end.timestamp() * 1_000_000)

# Filter metrics for the product-catalog service within the alert time window
alert_window_metrics = product_catalog_service_metrics[
    (product_catalog_service_metrics['time'] >= alert_start_us) &
    (product_catalog_service_metrics['time'] <= alert_end_us)
]

# Merge with P95 thresholds to identify anomalies
alert_window_metrics = alert_window_metrics.merge(kpi_p95_thresholds, on='metric', how='left')

# Identify anomalies where the value exceeds the P95 threshold
anomalies = alert_window_metrics[alert_window_metrics['value'] > alert_window_metrics['p95_threshold']]

anomalies

```
Out[5]:
```
During the alert time window (2026-04-13 04:59:52 to 2026-04-13 05:05:32 UTC+8), the following anomalies were identified for the `product-catalog` service where KPI values exceeded the global P95 thresholds:
- `cpu`: Exceeded the threshold of 0.02295 with values of 0.025296 and 0.023526.
- `workload`: Exceeded the threshold of 17,871 with a value of 18,449.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set entity_id      entity_name    metric         value metric_set_id service  p95_threshold
0  1776027592000000    apm  apm.service.legacy            product-catalog       cpu      0.025296          None    None       0.022954
3  1776027592000000    apm  apm.service.legacy            product-catalog  workload  18449.000000          None    None   17871.000000
4  1776027622000000    apm  apm.service.legacy            product-catalog       cpu      0.023526          None    None       0.022954```
```

In [ ]:
# In[6]:

# Ensure 'duration' column is numeric for aggregation
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')

# Filter traces related to the product-catalog service during the alert time window
product_catalog_traces = traces_data[
    (traces_data['serviceName'].str.contains('product-catalog', na=False)) &
    (traces_data['startTime'] >= alert_start_us * 1_000) &  # Convert microseconds to nanoseconds
    (traces_data['endTime'] <= alert_end_us * 1_000)
]

# Analyze traces to identify operations or downstream dependencies linked to anomalies
trace_analysis = product_catalog_traces.groupby(['spanName', 'serviceName'])['duration'].agg(['count', 'mean', 'max']).reset_index()
trace_analysis.columns = ['spanName', 'serviceName', 'trace_count', 'avg_duration_ns', 'max_duration_ns']

trace_analysis

```
Out[6]:
```
During the alert time window, the traces for the `product-catalog` service show the following operations:
- `GetProduct`: Executed 6,512 times with an average duration of 59,419 nanoseconds and a maximum duration of 418,733 nanoseconds.
- `ListProducts`: Executed 2,909 times with an average duration of 59,526 nanoseconds and a maximum duration of 341,841 nanoseconds.

These operations are likely contributing to the anomalies in `cpu` and `workload` due to their high execution counts and durations.

The original code execution output of IPython Kernel is also provided below for reference:

spanName      serviceName  trace_count  avg_duration_ns  max_duration_ns
0    oteldemo.ProductCatalogService/GetProduct  product-catalog         6512     59419.380989           418733
1  oteldemo.ProductCatalogService/ListProducts  product-catalog         2909     59526.123066           341841```
```

In [ ]:
# In[7]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Ensure '_time_' column is numeric for filtering
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs related to the product-catalog service during the alert time window
product_catalog_logs = logs_data[
    (logs_data['_container_name_'].str.contains('product-catalog', na=False)) &
    (logs_data['_time_'] >= alert_start_us) &
    (logs_data['_time_'] <= alert_end_us)
]

# Analyze logs for error messages or patterns
error_logs = product_catalog_logs[product_catalog_logs['content'].str.contains('error', case=False, na=False)]

error_logs

```
Out[7]:
```
During the alert time window, no error messages or patterns were found in the logs related to the `product-catalog` service. This suggests that the anomalies in `cpu` and `workload` are not directly linked to any logged errors.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__, __tag__:_node_name_]
Index: []

[0 rows x 15 columns]```
```